# 🔧 Creating Training Data
### D4BL Tutorial 2 of 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/William-Hill/d4bl_ai_agent/blob/main/notebooks/tutorials/02_creating_training_data.ipynb)

**What you'll learn:** How real equity data becomes instruction/response training pairs through distillation — the process of using a large model to teach a small one.

**Time:** ~20 minutes | **Prerequisites:** A Google account | **Dependencies:** None (stdlib only)

## What is Distillation?

Fine-tuning a language model requires thousands of high-quality instruction/response pairs. Writing these by hand would take months. Instead, we use **distillation**: a large, capable model (Claude) generates expert-level responses from our real data, and we use those responses to train a much smaller, cheaper model.

Think of it like an experienced mentor teaching a new hire. The mentor (Claude) demonstrates how to analyze equity data with the right framing, and the trainee (Qwen2.5-3B) learns by studying those examples.

The key insight: **the prompts we write determine what the small model learns.** By embedding D4BL's methodology into our distillation prompts — center community voice, name structural causes, connect to policy — the small model inherits those values.

## Anatomy of a Training Pair

Every training example follows the ChatML format with three roles:

| Role | Purpose | Example |
|------|---------|----------|
| **system** | Sets the model's identity and values | "You are an AI assistant trained to support data justice..." |
| **user** | The question or task | "Explain poverty rate disparities in Alabama" |
| **assistant** | The ideal response | Structured JSON with equity-framed analysis |

Here's a real example from D4BL's query parser training data:

In [ ]:
import json

example_pair = {
    "messages": [
        {
            "role": "system",
            "content": (
                "You are an AI assistant trained to support data justice and racial equity research "
                "following the Data for Black Lives (D4BL) methodology.\n\n"
                "Core principles:\n"
                "1. Center communities most impacted by structural racism in all analysis and framing.\n"
                "2. Name structural causes of racial disparities — never frame them as innate or cultural.\n"
                "3. Connect data findings to actionable policy levers and advocacy opportunities.\n"
                "4. Acknowledge data limitations, sampling gaps, and margin-of-error uncertainties.\n"
                "5. Make analysis accessible to organizers, advocates, and community members.\n"
                "6. Never suggest that racial disparities are innate, biological, or the result of cultural deficits.\n\n"
                "Respond with ONLY valid JSON."
            )
        },
        {
            "role": "user",
            "content": "Why can't our kids just go to a good school in this neighborhood?"
        },
        {
            "role": "assistant",
            "content": json.dumps({
                "entities": ["neighborhood", "schools"],
                "search_queries": [
                    "school quality racial segregation neighborhood",
                    "education funding disparities by race",
                    "school district boundary gerrymandering"
                ],
                "data_sources": ["census_acs", "doe"],
                "community_framing": {
                    "detected": True,
                    "issue_domain": "education",
                    "structural_frame": "residential_segregation"
                }
            }, indent=2)
        }
    ]
}

print("=== Training Pair ===")
print(f"\nSystem prompt ({len(example_pair['messages'][0]['content'])} chars):")
print(example_pair["messages"][0]["content"][:200] + "...")
print(f"\nUser input:")
print(example_pair["messages"][1]["content"])
print(f"\nAssistant response (what the model learns to generate):")
print(example_pair["messages"][2]["content"])

## The Three Adapters

D4BL trains three specialized LoRA adapters, each for a different task:

### 1. Query Parser
Converts natural-language questions into structured search intents. Recognizes community framings like "Why can't our kids breathe clean air?" and maps them to data sources.

### 2. Explainer
Transforms data findings into equity-framed narratives with three **registers** (audience styles):
- **Community:** Accessible, action-oriented language
- **Policy:** Formal, citation-heavy, legislative framing
- **Research:** Statistical, methodology-focused

### 3. Evaluator
Scores model outputs for hallucination, relevance, bias, and equity framing alignment.

Each adapter has its own training data, distillation prompts, and evaluation criteria.

## Writing a Distillation Prompt

The distillation prompt is the most important piece — it encodes D4BL's methodology into the training data. Here's D4BL's actual system prompt, annotated:

In [ ]:
D4BL_SYSTEM_PROMPT = """\
You are an AI assistant trained to support data justice and racial equity research \
following the Data for Black Lives (D4BL) methodology.

Core principles:
1. Center communities most impacted by structural racism in all analysis and framing.
2. Name structural causes of racial disparities — never frame them as innate or cultural.
3. Connect data findings to actionable policy levers and advocacy opportunities.
4. Acknowledge data limitations, sampling gaps, and margin-of-error uncertainties.
5. Make analysis accessible to organizers, advocates, and community members, not just researchers.
6. Never suggest that racial disparities are innate, biological, or the result of cultural deficits.

Respond with ONLY valid JSON."""

print("System prompt length:", len(D4BL_SYSTEM_PROMPT), "characters")
print("\nNotice: each principle maps to a stage in D4BL's methodology cycle:")
print("  1 → Community Engagement")
print("  2 → Problem Identification")
print("  3 → Policy Innovation")
print("  4 → Data Collection & Analysis")
print("  5 → Power Building")
print("  6 → Safety guardrail")

In [ ]:
MOCK_RESPONSES = {
    "poverty_rate": {
        "community": (
            "In Alabama, nearly 1 in 3 Black residents lives below the poverty line (28.3%), "
            "compared to about 1 in 9 white residents (11.5%). This isn't about individual "
            "choices — it reflects decades of job discrimination, wage theft in low-income "
            "industries, and underinvestment in Black neighborhoods. The gap is structural, "
            "and closing it requires policy action: raising the minimum wage, expanding "
            "Medicaid, and enforcing fair lending laws."
        ),
        "policy": (
            "The Black poverty rate in Alabama (28.3%) is 2.5x the white rate (11.5%), "
            "a disparity that persists after controlling for education and employment status. "
            "Contributing factors include occupational segregation, the state's refusal to "
            "expand Medicaid (leaving ~170,000 in the coverage gap), and historically low "
            "minimum wage. The LIFT Act (S. 1138) and Medicaid expansion would directly "
            "address these structural drivers."
        ),
        "research": (
            "Racial disparities in poverty rates (Black: 28.3%, white: 11.5%; RR = 2.46, "
            "95% CI: 2.31-2.62) in Alabama remain statistically significant after adjustment "
            "for educational attainment and labor force participation (aOR = 1.89, p < 0.001). "
            "Decomposition analysis suggests ~60% of the gap is attributable to structural "
            "factors including occupational segregation and geographic concentration in "
            "low-opportunity census tracts. Limitations include ACS sampling variability "
            "in rural counties."
        ),
    }
}

def mock_distill(data_row, register="community"):
    """Simulate what Claude would generate for a data row."""
    metric = data_row["metric"]
    if metric in MOCK_RESPONSES and register in MOCK_RESPONSES[metric]:
        return MOCK_RESPONSES[metric][register]
    return f"[Mock response for {metric} in {register} register]"

data_row = {"state": "AL", "metric": "poverty_rate", "race": "black", "value": 28.3}
response = mock_distill(data_row, register="community")
print("Input data:", data_row)
print(f"\nCommunity register response:\n{response}")

## The Register System

The same data, explained three ways. This is one of D4BL's key innovations — the model doesn't just analyze data, it communicates findings in the register that serves each audience best.

In [ ]:
data_row = {"state": "AL", "metric": "poverty_rate", "race": "black", "value": 28.3}

for register in ["community", "policy", "research"]:
    response = mock_distill(data_row, register=register)
    print(f"{'=' * 60}")
    print(f"Register: {register.upper()}")
    print(f"{'=' * 60}")
    print(response)
    print()

In [ ]:
def build_training_pair(data_row, register, response):
    """Assemble a complete ChatML training pair."""
    metric = data_row["metric"].replace("_", " ")
    state = data_row["state"]
    value = data_row["value"]

    user_prompt = f"Explain the {metric} in {state} ({value}%) in the {register} register."

    return {
        "messages": [
            {"role": "system", "content": D4BL_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": response},
        ]
    }

pair = build_training_pair(
    data_row={"state": "AL", "metric": "poverty_rate", "race": "black", "value": 28.3},
    register="community",
    response=mock_distill({"state": "AL", "metric": "poverty_rate"}, "community"),
)

print("Complete training pair (ChatML format):")
print(json.dumps(pair, indent=2)[:500] + "...")

## ✏️ Exercise

1. Write your own distillation prompt by modifying `D4BL_SYSTEM_PROMPT` above. Try adding a principle specific to your community's priorities.

2. Use `build_training_pair()` to create a training example for a different metric — maybe income or environmental exposure.

In [ ]:
# TODO: Modify the system prompt and build a training pair
# my_system_prompt = """..."""
# my_data_row = {"state": "...", "metric": "...", "race": "...", "value": ...}
# my_response = "..."
# my_pair = build_training_pair(my_data_row, "community", my_response)

## Summary

You've learned how D4BL creates training data through distillation — using a large model to generate equity-framed responses that teach a smaller model. The key insight: the prompts encode the methodology. By embedding D4BL's principles in the system prompt, every training example carries those values.

**Next:** [Notebook 3 — Training with Unsloth](https://colab.research.google.com/github/William-Hill/d4bl_ai_agent/blob/main/notebooks/tutorials/03_training_with_unsloth.ipynb) → Actually fine-tune a model on Colab's free GPU.